# Project 1: Looking at Age in the 2026 CPS-ASEC

In my first project for Computing in Context at Columbia SIPA, I'll be examining the mean, median, and mode age in the United States (U.S.) using U.S. Census Bureau survey data. More specifically, I'll be using the 2026 Current Population Survey (CPS) Annual Social and Economic Supplement (ASEC), which is a large, nationally representative survey conducted by the Census Bureau to produce poverty statistics. The 2026 CPS-ASEC, which was conducted in March 2026, is reflective of incomes and demographic trends in 2025. 

For this project, I retrieved the person-level 2026 CPS ASEC file from the Census website, which can be accessed [here.](https://www.census.gov/data/datasets/time-series/demo/cps/cps-asec.html) The results in this project will all be unweighted; that is, the mean, median, and mode ages calculated will be representative of the survey's sample rather than the full U.S. population.

To start, I'll open the dataset and limit the data frame to just three columns: the household sequence number (*PH_SEQ*), line number (*A_LINENO*), and age (*A_AGE*) for each respondent and their household members. Please note that the household sequence number is not personally-identifiable information; it's just a random, anonymized household identifier used to link observations in the CPS-ASEC household, family, and person files. I'll also take a glimpse at the first five observations in the data and check that each of the three columns are numeric.

In [47]:
# Import pandas library
import pandas as pd

# Read the 2026CPS ASEC CSV file and select the age column
asec26 = pd.read_csv("pppub26.csv", usecols=["PH_SEQ", "A_LINENO", "A_AGE"])

# Display the first few rows of the DataFrame
print(asec26.head())

# Check the data types of the columns
print(asec26.dtypes)

   PH_SEQ  A_LINENO  A_AGE
0       1         1     57
1       1         2     67
2       5         1     71
3       6         1     78
4      13         1     68
PH_SEQ      int64
A_LINENO    int64
A_AGE       int64
dtype: object


## Data Cleaning
Before I get ahead of myself and calculate the mean, median, and mode ages, I first want to remove top-coded age values. In the public-use CPS-ASEC data we're using, the Census Bureau top-codes exact ages to protect respondent privacy. Ages 80 through 84 are grouped into a single code (80), and ages 85 and older are grouped into another (85). Thus, keeping the values 80 and 85 in the data may result in artificial, or false, mode values of 80 or 85.

In [48]:
asec26 = asec26[~asec26["A_AGE"].isin([80, 85])].copy()

## Calculating the Mean, Median, and Mode Ages (The "Easy" Way)
Now that the data's cleaned, I'll calculate the mean, median, and mode ages in the U.S. using the *supposedly* "easy" way. That just means I'll use the Pandas standard library to calculate these values, which is fairly straightforward.

In [49]:
# Calculate mean, median, and mode of the age column
mean_age = asec26["A_AGE"].mean()
median_age = asec26["A_AGE"].median()
mode_ages = asec26["A_AGE"].mode()

# Print the results
print(f"Mean age: {mean_age:.2f}")
print(f"Median age: {median_age}")
print(f"Mode age(s): {mode_ages.tolist()}")

Mean age: 38.07
Median age: 38.0
Mode age(s): [16]


So looks like the *unweighted* mean, or average, age in the U.S. in 2025 was about 38! The median was also 38, and the mode was 16. I'm quite surprised 16-year-olds made up the most common age.

## Calculating the Mean, Median, and Mode Ages (The "Hard" Way)
Now because calculating the mean, median, and mode ages the previous way was too straightforward, I've been assigned a challenge. I now need to calculate the three statistics again manually without using the Python standard library, the statistics module, or anything very simple (or I risk losing 10 points). I'll use the same data as before, but I won't be able to access the data frame or use functions with the names mean, median, or mode. Let's try it out!

First, I'll open the data again and create a list with all the age values, excluding the top-coded values of 80 and 85. This list will be the basis for my calculations.

In [50]:
# Import the csv module
import csv

# Create a list to store ages
ages = []

# Read the CSV file and extract ages, excluding 80 and 85
with open("pppub26.csv", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        age = int(row["A_AGE"])
        if age not in (80, 85):
            ages.append(age)

### 1. Mean
Now, I'll calculate the mean, or average, age the hard way. I'll first create a variable called *total_age* that will store the sum of all the age values in the *ages* list we created. That is, the "for" loop here is summing each age in the *ages* list into the *total_age* variable. The mean is then calculated by dividing the total sum of ages (*total_age*) by the total number of age values in the *ages* list.

In [51]:
# Calculate the mean

# Create a variable to hold the total age
total_age = 0

# Calculate the total age by summing all ages in the list
for age in ages:
    total_age += age
hard_mean = total_age / len(ages)

# Print the mean
print(f"Mean age: {hard_mean:.2f}")

Mean age: 38.07


### 2. Median
Turning to the median, I'll again use the *ages* list we created. I'll first sort the *ages* list to have all the age values in ascending order, and I'll then store the age value in the middle of the list in a variable called *middle*. If the number of ages in the *ages* list is odd, I'll just identify the median as the middle value. If the number of ages in the list is even, I'll calculate the median as the average of the middle two ages.

In [52]:
# Calculate the median

# Sort the ages list to prepare for median calculation
ages.sort()

# Find the middle value of the sorted list
middle = len(ages) // 2

# Calculate the median based on whether the number of ages is odd or even
if len(ages) % 2 == 1:
    hard_median = ages[middle]
else:
    hard_median = (ages[middle - 1] + ages[middle]) / 2

# Print the median
print(f"Median age: {hard_median}")

Median age: 38


### 3. Mode
And finally, I'll calculate the mode the hard way. I'll first create a dictionary to store the number of times each age appears in the *ages* list. Each age will be a key in the dictionary, and the number of times that age appears in the *ages* list will be the value associated with that key. Then, I'll find the key(s) with the highest value(s) (i.e., the age(s) that appear(s) most frequently), and create a list containing these keys. The list, thus, will show us the mode age(s).

In [53]:
# Create a dictionary to count occurrences of each age
age_counts = {}

# Count ages and store the counts in the dictionary
for age in ages:
    age_counts[age] = age_counts.get(age, 0) + 1

# Find the highest count (the mode) and identify all ages with that count
highest_count = max(age_counts.values())

# Create a list of ages that have the highest count (the mode)
hard_modes = [age for age, count in age_counts.items() if count == highest_count]

# Print the mode(s)
print(f"Mode age(s): {hard_modes}")

Mode age(s): [16]


That wasn't too bad! And as you can see, we got the same results. The mean age was about 38, the median was also 38, and the mode age was 16. Now onto the last part!

## Data Visualization
The last step in this project is to create a data visualization of my variable of choice. For this part, I'll create a histogram showing the frequency distribution of *A_AGE*. To avoid having 80 bars, I'll first group the ages into five-year age bins (e.g., 0-4, 20-24, 45-49, etc.). I then find the number of observations in each five-year bin and store these counts in a list. I then set the width for each bar, obtain the maximum count of bins to scale my histogram, print the headers for the histogram, and finally create the figure. The figure is created using a "for" loop that adds each bin to the figure along with a label showing the number of observations who fall into each bin. 

The histogram shows us that most Americans fell into the 10-14, 35-39, 40-44, and 15-19 age groups in 2025. 

In [54]:
# Create a list of five-year age bins from 0 to 80
age_bins = range(0, 80, 5)

# Calculate the number of respondents in each five-year age bin
bin_counts = [
    int(((asec26["A_AGE"] >= age) & (asec26["A_AGE"] < age + 5)).sum())
    for age in age_bins
]

# Set the width of the bar for the histogram and find the maximum bin count for scaling
bar_width = 40
max_bin_count = max(bin_counts)

# Print the histogram title
print("Age Distribution (Five-Year Bins)")
print("Frequency (respondents; bar lengths are scaled to the largest bin)")
print("Age (years)")

# Create a histogram of the age distribution using five-year bins
for age, count in zip(age_bins, bin_counts):
    bar_length = round(count / max_bin_count * bar_width)
    bar = "█" * bar_length
    print(f"{age:02d}–{age + 4:02d} |{bar:<{bar_width}}| {count:,}")

Age Distribution (Five-Year Bins)
Frequency (respondents; bar lengths are scaled to the largest bin)
Age (years)
00–04 |██████████████████████████████          | 7,259
05–09 |████████████████████████████████████    | 8,704
10–14 |████████████████████████████████████████| 9,596
15–19 |███████████████████████████████████████ | 9,264
20–24 |█████████████████████████████           | 6,964
25–29 |███████████████████████████████         | 7,415
30–34 |████████████████████████████████████    | 8,692
35–39 |███████████████████████████████████████ | 9,340
40–44 |███████████████████████████████████████ | 9,291
45–49 |████████████████████████████████████    | 8,715
50–54 |████████████████████████████████        | 7,729
55–59 |███████████████████████████████         | 7,519
60–64 |██████████████████████████████████      | 8,039
65–69 |██████████████████████████████████      | 8,077
70–74 |█████████████████████████████           | 6,943
75–79 |██████████████████████                  | 5,374
